# 🌿 PlantDx — End-to-End Demo

This notebook walks through the full PlantDx workflow:
1. Install and import the library
2. Load a pre-trained checkpoint
3. Run inference on a leaf image
4. Visualise the prediction with confidence bars
5. Generate a Grad-CAM explanation heatmap

**Requirements:** `pip install -e ".[gradcam]"` from the repo root.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path
from PIL import Image

from plantdx import PlantDiseaseClassifier

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {"CUDA" if torch.cuda.is_available() else "CPU"}')

## 1. Load checkpoint

Download the latest weights from [GitHub Releases](https://github.com/Pelex04/Diagnosis-plant-model/releases/latest)
and place them at `checkpoints/best_model.pth`.

In [ ]:
CHECKPOINT = Path('../checkpoints/best_model.pth')

if not CHECKPOINT.exists():
    raise FileNotFoundError(
        f'Checkpoint not found at {CHECKPOINT}.\n'
        'Download it from: https://github.com/Pelex04/Diagnosis-plant-model/releases/latest'
    )

clf = PlantDiseaseClassifier.from_checkpoint(CHECKPOINT)
print(f'Loaded {len(clf.class_names)} classes:')
for i, name in enumerate(clf.class_names):
    print(f'  {i:2d}  {name}')

## 2. Run inference on a sample image

In [ ]:
IMAGE_PATH = Path('../docs/assets/sample_leaf.jpg')

results = clf.predict(IMAGE_PATH, top_k=5)

print('Top-5 predictions:')
print('-' * 55)
for rank, r in enumerate(results, 1):
    bar = '█' * int(r['confidence'] * 40)
    print(f'  {rank}. {r["class"]:<42} {r["confidence"]*100:5.1f}%  {bar}')

## 3. Visualise predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: original image
img = Image.open(IMAGE_PATH).convert('RGB')
axes[0].imshow(img)
axes[0].set_title('Input Image', fontsize=13)
axes[0].axis('off')

# Right: horizontal confidence bar chart
top5_labels = [r['class'].replace('_', ' ') for r in results]
top5_confs  = [r['confidence'] * 100 for r in results]
colors      = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(results))]

bars = axes[1].barh(top5_labels[::-1], top5_confs[::-1], color=colors[::-1], height=0.6)
axes[1].set_xlabel('Confidence (%)', fontsize=12)
axes[1].set_title('Top-5 Predictions', fontsize=13)
axes[1].set_xlim(0, 105)
axes[1].tick_params(axis='y', labelsize=10)
axes[1].grid(axis='x', alpha=0.3)

for bar, conf in zip(bars, top5_confs[::-1]):
    axes[1].text(conf + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{conf:.1f}%', va='center', fontsize=10)

plt.suptitle(f'PlantDx Prediction — {IMAGE_PATH.name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/prediction_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/prediction_demo.png')

## 4. Grad-CAM Explanation

Shows **which part of the leaf** the model focused on.
Requires: `pip install grad-cam`

In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from plantdx import get_val_transform
    GRADCAM_AVAILABLE = True
except ImportError:
    print('grad-cam not installed. Run: pip install grad-cam')
    GRADCAM_AVAILABLE = False

if GRADCAM_AVAILABLE:
    transform    = get_val_transform()
    pil_image    = Image.open(IMAGE_PATH).convert('RGB')
    input_tensor = transform(pil_image).unsqueeze(0).to(clf.device)
    img_array    = np.array(pil_image.resize((380, 380))).astype(np.float32) / 255.0

    top_class_idx = clf.class_names.index(results[0]['class'])
    target_layer  = clf.model.features[-1]

    clf.model.eval()
    with GradCAM(model=clf.model, target_layers=[target_layer]) as cam:
        grayscale_cam = cam(
            input_tensor=input_tensor,
            targets=[ClassifierOutputTarget(top_class_idx)]
        )[0]

    overlay = show_cam_on_image(img_array, grayscale_cam, use_rgb=True, image_weight=0.5)

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(np.array(pil_image.resize((380, 380))))
    axes[0].set_title('Original', fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(overlay)
    axes[1].set_title(
        f'Grad-CAM\n{results[0]["class"]} ({results[0]["confidence"]*100:.1f}%)',
        fontsize=11
    )
    axes[1].axis('off')

    plt.suptitle('Grad-CAM Explanation — model attention heatmap', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/gradcam_demo.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → reports/gradcam_demo.png')

## 5. Batch inference

Run predictions across multiple images and display a summary grid.

In [ ]:
image_dir = Path('../data/')
extensions = {'.jpg', '.jpeg', '.png'}

images = sorted([
    p for p in image_dir.rglob('*')
    if p.suffix.lower() in extensions
])[:12]  # limit to 12 for display

print(f'Running inference on {len(images)} images...')
batch_results = clf.predict_batch(images, top_k=1)

cols = 4
rows = (len(images) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
axes = axes.flatten()

for ax, img_path, preds in zip(axes, images, batch_results):
    img  = Image.open(img_path).convert('RGB')
    pred = preds[0] if preds else {'class': 'unknown', 'confidence': 0.0}
    ax.imshow(img)
    ax.set_title(
        f"{pred['class'].replace('_', ' ')}\n{pred['confidence']*100:.1f}%",
        fontsize=9
    )
    ax.axis('off')

for ax in axes[len(images):]:
    ax.axis('off')

plt.suptitle('PlantDx Batch Inference', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()